# Pipeline

In [1]:
!pip install mlflow dagshub -q
!pip install imbalanced-learn -q
!pip install shap -q
!pip install lightgbm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 106.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import numpy as np
import pandas as pd
import dagshub
import mlflow
import mlflow.sklearn

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None


dagshub.init(repo_owner='Sula1909', repo_name='ML-Assignment2', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/Sula1909/ML-Assignment2.mlflow')
mlflow.set_experiment('RandomForest_Training')



class ColumnRenameTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.rename_map_ = {c: c.replace('-', '_') for c in X.columns if '-' in c}
        return self

    def transform(self, X):
        X_out = X.copy()
        if self.rename_map_:
            X_out = X_out.rename(columns=self.rename_map_)
        return X_out


def reduce_mem_usage(df):
    df = df.copy()
    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_datetime64_any_dtype(col_type):
            continue

        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if pd.api.types.is_integer_dtype(col_type):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    return df


class MemoryReductionTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return reduce_mem_usage(X)


class OutlierRemovalTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, amt_col='TransactionAmt', threshold=30000, drop_on_inference=False):
        self.amt_col = amt_col
        self.threshold = threshold
        self.drop_on_inference = drop_on_inference

    def fit(self, X, y=None):
        return self

    def fit_transform(self, X, y=None, **fit_params):
        self.fit(X, y)
        X_out = X.copy()
        if self.amt_col in X_out.columns:
            mask = X_out[self.amt_col].fillna(-np.inf) <= self.threshold
            X_out = X_out.loc[mask].copy()
        return X_out

    def transform(self, X):
        X_out = X.copy()
        if self.drop_on_inference and self.amt_col in X_out.columns:
            mask = X_out[self.amt_col].fillna(-np.inf) <= self.threshold
            X_out = X_out.loc[mask].copy()
        return X_out


class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, transactiondt_col='TransactionDT', amt_col='TransactionAmt', card1_col='card1'):
        self.transactiondt_col = transactiondt_col
        self.amt_col = amt_col
        self.card1_col = card1_col

    def fit(self, X, y=None):
        if self.card1_col in X.columns and self.amt_col in X.columns:
            self.card1_amt_mean_ = X.groupby(self.card1_col)[self.amt_col].mean()
        else:
            self.card1_amt_mean_ = pd.Series(dtype=np.float64)
        return self

    def transform(self, X):
        X_out = X.copy()

        if self.transactiondt_col in X_out.columns:
            t = X_out[self.transactiondt_col].fillna(0)
            X_out['day'] = (t // (24 * 60 * 60)).astype(np.int32)
            X_out['hour'] = ((t // 3600) % 24).astype(np.int8)

            def hour_feature(hour):
                if hour > 4 and hour < 12:
                    return 'highalert'
                if hour == 12 or hour == 19:
                    return 'lowalert'
                if hour == 3 or hour == 4 or hour == 24:
                    return 'mediumalert'
                return 'noalert'

            X_out['hour_alertFeature'] = X_out['hour'].apply(hour_feature)
        else:
            X_out['day'] = 0
            X_out['hour'] = 0
            X_out['hour_alertFeature'] = 'noalert'

        if self.card1_col in X_out.columns and self.amt_col in X_out.columns and len(self.card1_amt_mean_) > 0:
            mean_amt = X_out[self.card1_col].map(self.card1_amt_mean_)
            ratio = X_out[self.amt_col] / mean_amt
            X_out['TransactionAmt_to_mean_card1'] = ratio.replace([np.inf, -np.inf], np.nan)
        else:
            X_out['TransactionAmt_to_mean_card1'] = np.nan

        return X_out


class CorrelationFilterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, prefix, threshold=0.95):
        self.prefix = prefix
        self.threshold = threshold

    def fit(self, X, y=None):
        target_cols = [c for c in X.columns if c.startswith(self.prefix)]
        target_cols = [c for c in target_cols if pd.api.types.is_numeric_dtype(X[c])]

        self.drop_cols_ = []
        if len(target_cols) > 1:
            corr = X[target_cols].corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            self.drop_cols_ = [col for col in upper.columns if any(upper[col] > self.threshold)]
        return self

    def transform(self, X):
        X_out = X.copy()
        to_drop = [c for c in self.drop_cols_ if c in X_out.columns]
        if to_drop:
            X_out = X_out.drop(columns=to_drop)
        return X_out


class LGBMFeatureSelectionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, prefix='V', importance_threshold=3, random_state=42):
        self.prefix = prefix
        self.importance_threshold = importance_threshold
        self.random_state = random_state

    def fit(self, X, y=None):
        self.drop_cols_ = []

        if y is None:
            return self

        v_cols = [c for c in X.columns if c.startswith(self.prefix)]
        v_cols = [c for c in v_cols if pd.api.types.is_numeric_dtype(X[c])]

        if len(v_cols) == 0:
            return self

        if LGBMClassifier is None:
            return self

        model = LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=self.random_state,
            n_jobs=-1
        )

        y_fit = y.loc[X.index] if isinstance(y, pd.Series) else y
        X_fit = X[v_cols].fillna(-999)
        model.fit(X_fit, y_fit)

        importances = pd.Series(model.feature_importances_, index=v_cols)
        self.drop_cols_ = importances[importances < self.importance_threshold].index.tolist()
        return self

    def transform(self, X):
        X_out = X.copy()
        to_drop = [c for c in self.drop_cols_ if c in X_out.columns]
        if to_drop:
            X_out = X_out.drop(columns=to_drop)
        return X_out


class FrequencyEncodingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        self.freq_maps_ = {}
        for col in self.columns:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(dropna=False, normalize=True)
        return self

    def transform(self, X):
        X_out = X.copy()
        for col, mapping in self.freq_maps_.items():
            X_out[f'{col}_freq_enc'] = X_out[col].map(mapping).astype(np.float32)
        return X_out


class WOEEncodingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns, event=1, smoothing=0.5):
        self.columns = columns
        self.event = event
        self.smoothing = smoothing

    def fit(self, X, y=None):
        if y is None:
            raise ValueError('WOEEncodingTransformer requires y in fit().')

        y_series = y.loc[X.index] if isinstance(y, pd.Series) else pd.Series(y, index=X.index)

        event_total = (y_series == self.event).sum()
        non_event_total = (y_series != self.event).sum()

        self.woe_maps_ = {}
        self.default_woe_ = {}

        for col in self.columns:
            if col not in X.columns:
                continue

            tmp = pd.DataFrame({
                'x': X[col].astype(str).fillna('__MISSING__'),
                'y': y_series
            })

            grouped = tmp.groupby('x')['y'].agg(['count', 'sum'])
            grouped['event_count'] = grouped['sum']
            grouped['non_event_count'] = grouped['count'] - grouped['sum']

            grouped['event_dist'] = (grouped['event_count'] + self.smoothing) / (event_total + self.smoothing * len(grouped))
            grouped['non_event_dist'] = (grouped['non_event_count'] + self.smoothing) / (non_event_total + self.smoothing * len(grouped))
            grouped['woe'] = np.log(grouped['event_dist'] / grouped['non_event_dist'])

            self.woe_maps_[col] = grouped['woe'].to_dict()
            self.default_woe_[col] = float(grouped['woe'].mean())

        return self

    def transform(self, X):
        X_out = X.copy()
        for col, mapping in self.woe_maps_.items():
            as_str = X_out[col].astype(str).fillna('__MISSING__')
            X_out[col] = as_str.map(mapping).fillna(self.default_woe_[col]).astype(np.float32)
        return X_out


class LowCardinalityOHETransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_unique=5):
        self.max_unique = max_unique

    def fit(self, X, y=None):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.ohe_cols_ = [c for c in cat_cols if X[c].nunique(dropna=False) <= self.max_unique]

        dummies = pd.get_dummies(
            X[self.ohe_cols_],
            columns=self.ohe_cols_,
            dummy_na=True,
            dtype=np.uint8
        ) if self.ohe_cols_ else pd.DataFrame(index=X.index)

        self.dummy_cols_ = dummies.columns.tolist()
        return self

    def transform(self, X):
        X_out = X.copy()

        if self.ohe_cols_:
            dummies = pd.get_dummies(
                X_out[self.ohe_cols_],
                columns=self.ohe_cols_,
                dummy_na=True,
                dtype=np.uint8
            )
            dummies = dummies.reindex(columns=self.dummy_cols_, fill_value=0)
            X_out = X_out.drop(columns=[c for c in self.ohe_cols_ if c in X_out.columns])
            X_out = pd.concat([X_out, dummies], axis=1)

        return X_out


class HighCardinalityLabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_unique=6):
        self.min_unique = min_unique

    def fit(self, X, y=None):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.label_cols_ = [c for c in cat_cols if X[c].nunique(dropna=False) >= self.min_unique]
        self.label_maps_ = {}

        for col in self.label_cols_:
            values = pd.Series(X[col].astype(str).fillna('__MISSING__').unique())
            self.label_maps_[col] = {v: i for i, v in enumerate(values)}

        return self

    def transform(self, X):
        X_out = X.copy()

        for col in self.label_cols_:
            if col in X_out.columns:
                as_str = X_out[col].astype(str).fillna('__MISSING__')
                X_out[col] = as_str.map(self.label_maps_[col]).fillna(-1).astype(np.int32)

        return X_out


class MedianImputerTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.medians_ = X.median(numeric_only=True)
        return self

    def transform(self, X):
        X_out = X.copy()
        num_cols = X_out.select_dtypes(include=[np.number]).columns
        X_out[num_cols] = X_out[num_cols].fillna(self.medians_)
        return X_out


class IndexAlignedRandomForestClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        n_estimators=500,
        max_depth=12,
        min_samples_split=50,
        min_samples_leaf=50,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.class_weight = class_weight
        self.random_state = random_state
        self.n_jobs = n_jobs

    def _build_model(self):
        return RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            class_weight=self.class_weight,
            random_state=self.random_state,
            n_jobs=self.n_jobs
        )

    def fit(self, X, y):
        self.model_ = self._build_model()

        if isinstance(y, pd.Series):
            y_fit = y.loc[X.index] if len(y) != len(X) else y
        else:
            y_fit = y

        self.model_.fit(X, y_fit)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


freq_cols = [
    'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain'
]

woe_cols = [
    'P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33', 'DeviceInfo'
]

fraud_pipeline = Pipeline(steps=[
    ('rename_cols', ColumnRenameTransformer()),
    ('reduce_mem', MemoryReductionTransformer()),
    ('remove_outliers', OutlierRemovalTransformer(amt_col='TransactionAmt', threshold=30000)),
    ('feature_engineering', FeatureEngineeringTransformer()),
    ('corr_filter_c', CorrelationFilterTransformer(prefix='C', threshold=0.95)),
    ('corr_filter_d', CorrelationFilterTransformer(prefix='D', threshold=0.95)),
    ('lgbm_v_selector', LGBMFeatureSelectionTransformer(prefix='V', importance_threshold=3, random_state=42)),
    ('freq_encoding', FrequencyEncodingTransformer(columns=freq_cols)),
    ('woe_encoding', WOEEncodingTransformer(columns=woe_cols, event=1, smoothing=0.5)),
    ('ohe_low_cardinality', LowCardinalityOHETransformer(max_unique=5)),
    ('label_encode_high_cardinality', HighCardinalityLabelEncoderTransformer(min_unique=6)),
    ('median_impute', MedianImputerTransformer()),
    ('rf_model', IndexAlignedRandomForestClassifier(
        n_estimators=500,
        max_depth=12,
        min_samples_split=50,
        min_samples_leaf=50,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])


# Raw merged dataframe (no manual preprocessing)
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
raw_train_merged = train_transaction.merge(train_identity, how='left', on='TransactionID')

X = raw_train_merged.drop(columns=['isFraud'])
y = raw_train_merged['isFraud']

if 'TransactionDT' not in X.columns:
    raise ValueError('TransactionDT is required for chronological split.')

sorted_idx = X.sort_values('TransactionDT').index
n_rows = len(sorted_idx)
train_end = int(n_rows * 0.70)
valid_end = int(n_rows * 0.85)

train_idx = sorted_idx[:train_end]
valid_idx = sorted_idx[train_end:valid_end]
test_idx = sorted_idx[valid_end:]

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_valid, y_valid = X.loc[valid_idx], y.loc[valid_idx]
X_test, y_test = X.loc[test_idx], y.loc[test_idx]

fraud_pipeline.fit(X_train, y_train)
train_pred = fraud_pipeline.predict_proba(X_train)[:, 1]
valid_pred = fraud_pipeline.predict_proba(X_valid)[:, 1]
test_pred = fraud_pipeline.predict_proba(X_test)[:, 1]

train_label_pred = fraud_pipeline.predict(X_train)
valid_label_pred = fraud_pipeline.predict(X_valid)
test_label_pred = fraud_pipeline.predict(X_test)

train_auc = roc_auc_score(y_train, train_pred)
valid_auc = roc_auc_score(y_valid, valid_pred)
test_auc = roc_auc_score(y_test, test_pred)

valid_precision = precision_score(y_valid, valid_label_pred, zero_division=0)
test_precision = precision_score(y_test, test_label_pred, zero_division=0)

valid_recall = recall_score(y_valid, valid_label_pred, zero_division=0)
test_recall = recall_score(y_test, test_label_pred, zero_division=0)

valid_f1 = f1_score(y_valid, valid_label_pred, zero_division=0)
test_f1 = f1_score(y_test, test_label_pred, zero_division=0)

print(f'Train AUC: {train_auc:.6f}')
print(f'Validation AUC: {valid_auc:.6f}')
print(f'Test AUC: {test_auc:.6f}')
print(" ")
print(f'Precision: {valid_precision:.6f} | Recall: {valid_recall:.6f} | F1: {valid_f1:.6f}')


mlflow_params = {
    'n_estimators': 500,
    'max_depth': 12,
    'min_samples_split': 50,
    'min_samples_leaf': 50,
    'max_features': 'sqrt',
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
}

with mlflow.start_run(run_name='RandomForest_pipeline'):
    mlflow.log_params(mlflow_params)

    mlflow.log_metric('Train_AUC', float(train_auc))
    mlflow.log_metric('Validation_AUC', float(valid_auc))
    mlflow.log_metric('Test_AUC', float(test_auc))

    mlflow.log_metric("Precision", valid_precision)
    mlflow.log_metric("Recall", valid_recall)
    mlflow.log_metric("F1", valid_f1)

    mlflow.sklearn.log_model(sk_model=fraud_pipeline, artifact_path='model')


Initialized MLflow to track repo "Sula1909/ML-Assignment2"

Repository Sula1909/ML-Assignment2 initialized!

[LightGBM] [Info] Number of positive: 14538, number of negative: 398838
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.551159 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23386
[LightGBM] [Info] Number of data points in the train set: 413376, number of used features: 337
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311789
[LightGBM] [Info] Start training from score -3.311789
Train AUC: 0.918909
Validation AUC: 0.872332
Test AUC: 0.874163
 
Precision: 0.185763 | Recall: 0.681131 | F1: 0.291913


2026/05/04 15:55:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 15:55:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_pipeline at: https://dagshub.com/Sula1909/ML-Assignment2.mlflow/#/experiments/1/runs/f0e04334c047418bafc4cadf3ff49f81
🧪 View experiment at: https://dagshub.com/Sula1909/ML-Assignment2.mlflow/#/experiments/1


# Intro

In [ ]:
!pip install mlflow dagshub -q

import numpy as np
import pandas as pd
import os
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    return df

In [ ]:
import dagshub
dagshub.init(repo_owner='Sula1909', repo_name='ML-Assignment2', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/Sula1909/ML-Assignment2.mlflow")

mlflow.set_experiment("RandomForest_Training")

In [ ]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.expand_frame_repr', False)

In [ ]:
print("Loading data...")
train_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

test_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = pd.merge(train_trans, train_id, on='TransactionID', how='left')
test = pd.merge(test_trans, test_id, on='TransactionID', how='left')
test.columns = test.columns.str.replace('-', '_')

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

del train_trans, train_id, test_trans, test_id
print(f"Dataset shape: {train.shape}")
print(f"Dataset shape: {test.shape}")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
na_rate = train.isna().mean()
print(na_rate[na_rate > 0.1].sort_values(ascending=False).to_string())

# EDA & CLEANING

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

null_data = train.isnull().mean().sort_values(ascending=False)[:50]

plt.figure(figsize=(15, 8))
null_data.plot(kind='bar', color='salmon')
plt.title('Top 50 Features with Most Missing Values (%)', fontsize=15)
plt.ylabel('Percentage of Nulls')
plt.axhline(y=0.90, color='r', linestyle='--', label='90% Threshold')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x=train['isFraud'], palette='viridis')
plt.title('Target Distribution (0: Normal, 1: Fraud)', fontsize=15)
plt.yscale('log') # Log Scale
plt.show()

print(f"Fraud Percentage: {train['isFraud'].mean()*100:.2f}%")

In [ ]:
train.head()

***> TransactionDT Analysis***

In [ ]:
print(train['TransactionDT'].dtype)

print(train["TransactionDT"].isnull().sum())

print(train["TransactionDT"].nunique())

In [ ]:
train[['TransactionDT']].describe().astype(int)

In [ ]:
plt.figure(figsize = (15,5))
sns.distplot(train['TransactionDT'],kde = False,bins=60)
sns.distplot(test['TransactionDT'],kde = False,bins=60)
plt.legend(['train', 'test'])
plt.ylabel('Frequency')

plt.title('Distribution of TransactiondDT');
plt.show()

***> TransactionAmt Analysis***

In [ ]:
print(train['TransactionAmt'].dtype)

print(train["TransactionAmt"].nunique())

print(train["TransactionAmt"].isnull().sum())
print(test["TransactionAmt"].isnull().sum())

In [ ]:
print(train['TransactionAmt'].describe())
print(" ")
print(test['TransactionAmt'].describe())

In [ ]:
train[train.TransactionAmt > 30000]

***> ProductCD Analysis***

In [ ]:
print(train["ProductCD"].dtype)

print(train["ProductCD"].nunique())

print(train["ProductCD"].isnull().sum())
print(test["ProductCD"].isnull().sum())

In [ ]:
train["ProductCD"].describe()

In [ ]:
train["ProductCD"].value_counts()

In [ ]:
plt.figure(figsize=(15,6))
plt.subplot(1,2,1)
train_ProductCD = train['ProductCD'].value_counts(normalize=True).mul(100).rename('percentage')\
.reset_index()
sns.barplot(x = "ProductCD", y = "percentage", data = train_ProductCD, palette = 'pastel')

plt.subplot(1,2,2)
test_ProductCD = test['ProductCD'].value_counts(normalize=True).mul(100).rename('percentage')\
.reset_index()
sns.barplot(x = "ProductCD", y = "percentage", data = test_ProductCD, palette = 'pastel')

plt.show();

In [ ]:
plt.figure(figsize = (10,6))

train_ProductCD = (train.groupby(['isFraud'])['ProductCD'].value_counts(normalize = True).rename('percentage').mul(100).reset_index().sort_values('ProductCD'))

sns.barplot(x = "ProductCD", y = "percentage", hue = "isFraud", data = train_ProductCD, palette = 'pastel')

plt.title('Percentage of fraud & legit across ProductCD in Train dataset')

plt.show();

***> DeviceType & DeviceInfo Analysis***

In [ ]:
print(train["DeviceType"].dtype)

print(train['DeviceType'].isnull().sum())

print(train['DeviceType'].unique())

In [ ]:
train['DeviceType'].describe()

In [ ]:
train['DeviceType'].value_counts()

In [ ]:
plt.figure(figsize = (18,8))

plt.subplot(1,2,1)

train_DeviceType = (train[~train['DeviceType'].isnull()].groupby(['isFraud'])['DeviceType'].value_counts(normalize=True).rename('percentage').mul(100).reset_index().sort_values('DeviceType'))
sns.barplot(x = "DeviceType", y = "percentage", hue = "isFraud", data = train_DeviceType, palette = 'pastel')
plt.title('Train Device Type')

plt.subplot(1,2,2)

test_DeviceType = test[~test['DeviceType'].isnull()]['DeviceType'].value_counts(normalize = True).mul(100).rename('percentage').reset_index()

sns.barplot( x = "DeviceType", y = "percentage", data=test_DeviceType, palette = 'pastel')
plt.xlabel('Device Type')
plt.title('Test')

plt.show()

In [ ]:
print(train["DeviceInfo"].dtype)

print(train['DeviceInfo'].isnull().sum())

print(train['DeviceInfo'].unique())

In [ ]:
train["DeviceInfo"].describe()

In [ ]:
train['DeviceInfo'].value_counts()[:5]

In [ ]:
train.groupby('DeviceInfo') \
    .count()['TransactionID'] \
    .sort_values(ascending = False) \
    .head(20) \
    .plot(kind = 'barh', figsize = (15 , 5), title = 'TOP 20 Devices')
    
plt.show()

# Feature Engineering

In [ ]:
train['day'] = ((train['TransactionDT'] //(3600*24)-1)%7)+1

test['day'] = ((test['TransactionDT'] //(3600*24)-1)%7)+1

In [ ]:
train_days = (train.groupby(['isFraud'])['day']
                     .value_counts(normalize=True)
                     .rename('Percentage')
                     .mul(100)
                     .reset_index()
                     .sort_values('day'))
sns.barplot(x="day", y = "Percentage", hue="isFraud", data = train_days, palette = 'pastel')
plt.show();

In [ ]:
train['hour'] = ((train['TransactionDT']//3600)%24)+1

test['hour'] = ((test['TransactionDT']//3600)%24)+1

In [ ]:
train_hour = (train.groupby(['isFraud'])['hour']
                     .value_counts(normalize = True)
                     .rename('Percentage')
                     .mul(100)
                     .reset_index()
                     .sort_values('hour'))
sns.barplot(x = "hour", y = "Percentage", hue = "isFraud", data = train_hour, palette = 'pastel')
plt.show();

In [ ]:
def hourFeature(hour):
    if hour > 4 and hour < 12:
        return "highalert"
    if hour ==12 or hour==19:
        return "lowalert"
    if hour==3 or hour==4 or hour==24:
        return "mediumalert"
    else:
        return "noalert"


train['hour_alertFeature'] = train['hour'].apply(hourFeature)

test['hour_alertFeature'] = test['hour'].apply(hourFeature)

In [ ]:
plt.figure(figsize = (15,6))
plt.subplot(1,2,1)
g1 = sns.scatterplot(x = "TransactionDT", y = "TransactionAmt", hue = "isFraud", data = train, alpha = 0.8, hue_order = [0,1])

plt.title('TransactionDT vs TransactionAmt by isFraud - Train')
plt.subplot(1,2,2)
sns.scatterplot(x = "TransactionDT",y="TransactionAmt", data=test, alpha=0.8, hue_order = [0,1])

plt.title('TransactionDT vs TransactionAmt by isFraud - Test')
plt.show()

In [ ]:
# Remove 2 TransactionAmt Outliers.
train = train[train['TransactionAmt'] < 30000]

In [ ]:
train.head()

In [ ]:
train.tail()

In [ ]:
c_cols = [f'C{i}' for i in range(1, 15) if f'C{i}' in train.columns]

c_nan_stats = (train[c_cols].isna().mean() * 100).reset_index()
c_nan_stats.columns = ['Feature', 'NaN_Percentage']

c_nan_stats = c_nan_stats.sort_values(by='NaN_Percentage', ascending=False)

print("C Features Missing Values Analysis:")
display(c_nan_stats)

In [ ]:
cor_c = train[c_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(cor_c, annot=True, fmt=".2f", cmap='RdBu_r', center=0)
plt.title('Correlation Matrix of C Features', fontsize=18)
plt.show()

In [ ]:
upper_c = cor_c.abs().where(np.triu(np.ones(cor_c.shape), k=1).astype(bool))

# Corr > 0.95 Threshold
to_drop_c = [column for column in upper_c.columns if any(upper_c[column] > 0.95)]


train.drop(columns=to_drop_c, inplace=True)
test.drop(columns=to_drop_c, inplace=True)

print(f"Deleted C Features: {to_drop_c}")

In [ ]:
d_cols = [f'D{i}' for i in range(1, 16) if f'D{i}' in train.columns]

d_nan_stats = (train[d_cols].isna().mean() * 100).reset_index()
d_nan_stats.columns = ['Feature', 'NaN_Percentage']

# Correlation with target
d_correlations = train[d_cols + ['isFraud']].corr()['isFraud'].reset_index()
d_correlations.columns = ['Feature', 'Correlation']

d_summary = pd.merge(d_nan_stats, d_correlations, on='Feature').sort_values(by='NaN_Percentage', ascending=False)

print("D Features: Missing Values & Correlation Analysis")
display(d_summary)

In [ ]:
import seaborn as sns

cor = train[d_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(cor, 
            annot=True,         
            fmt=".2f",           
            cmap='RdBu_r',       
            vmin=-1, vmax=1,   
            center=0,
            linewidths=0.5,      
            linecolor='white')  

plt.title('Correlation Matrix of D Features', fontsize=18)
plt.show()

In [ ]:
upper = cor.abs().where(np.triu(np.ones(cor.shape), k=1).astype(bool))

# Corr > 0.95 threshold
to_drop_d = [column for column in upper.columns if any(upper[column] > 0.95)]

print(f"Deleted D Features: {to_drop_d}")

train.drop(columns=to_drop_d, inplace=True)
test.drop(columns=to_drop_d, inplace=True)

In [ ]:
v_cols = [f'V{i}' for i in range(1, 15)]
plt.figure(figsize=(12, 10))
sns.heatmap(train[v_cols].corr(), annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap of V1-V14 Features', fontsize=15)
plt.show()

In [ ]:
# Removing useless V columns with LGBM

from lightgbm import LGBMClassifier
import pandas as pd

v_features = [c for c in train.columns if c.startswith('V')]

split_idx = int(train.shape[0] * 0.70)

X_v_train = train[v_features].iloc[:split_idx]
y_v_train = train['isFraud'].iloc[:split_idx]
X_v_val = train[v_features].iloc[split_idx:]
y_v_val = train['isFraud'].iloc[split_idx:]

clf = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1 
)

clf.fit(X_v_train, y_v_train)

v_importance = pd.DataFrame({
    'feature': v_features,
    'importance': clf.feature_importances_
})

v_to_remove = v_importance[v_importance['importance'] < 3]['feature'].tolist()

print(f"Total number of V columns: {len(v_features)}")
print(f"V columns to delete: {len(v_to_remove)}")
print(f"Needed V columns: {len(v_features) - len(v_to_remove)}")

train.drop(columns=v_to_remove, inplace=True)
test.drop(columns=v_to_remove, inplace=True)


In [ ]:
# To catch sudden big amount transaction from a card

train['card1_amt_mean'] = train.groupby('card1')['TransactionAmt'].transform('mean')
test['card1_amt_mean'] = test.groupby('card1')['TransactionAmt'].transform('mean')

# Ratio of transaction
train['TransactionAmt_to_mean_card1'] = train['TransactionAmt'] / train['card1_amt_mean']
test['TransactionAmt_to_mean_card1'] = test['TransactionAmt'] / test['card1_amt_mean']

train.drop('card1_amt_mean', axis=1, inplace=True)
test.drop('card1_amt_mean', axis=1, inplace=True)

print("New Feature 'TransactionAmt_to_mean_card1'")

In [ ]:
id_cols = [c for c in train.columns if c.startswith('id_')]

id_na_stats = train[id_cols].isna().mean()

print(id_na_stats)

In [ ]:
train.isin([np.inf, -np.inf]).sum().any()

In [ ]:
# Frequency Encoding 

freq_cols = ['card1', 'card2', 'card3', 'card5', 
             'addr1', 'addr2', 
             'P_emaildomain', 'R_emaildomain']

for col in freq_cols:
    if col in train.columns:
        # Count frequencies from TRAIN only
        mapping = train[col].astype('object').value_counts(dropna=False)
        
        train[col + '_freq'] = train[col].astype('object').map(mapping).astype('int32')
        
        test[col + '_freq'] = test[col].astype('object').map(mapping).fillna(0).astype('int32')
        
        print(f"Frequency encoding finished: {col}")

train = train.copy()
test = test.copy()

In [ ]:
# WOE & IV Implementation
# Applied to high amount of variable categorical columns

woe_cols = ['P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33', 'DeviceInfo']

def calculate_woe_iv(df, feature, target):
    """Calculate WOE and IV for a categorical feature."""
    df = df.copy()
    df[feature] = df[feature].astype(str).fillna('Missing')
    
    total_events = df[target].sum()        # total fraud cases
    total_non_events = (df[target] == 0).sum()  # total normal cases
    
    grouped = df.groupby(feature)[target].agg(['sum', 'count'])
    grouped.columns = ['events', 'total']
    grouped['non_events'] = grouped['total'] - grouped['events']
    
    grouped['dist_events'] = grouped['events'] / total_events
    grouped['dist_non_events'] = grouped['non_events'] / total_non_events
    
    # Avoid log(0) by clipping
    grouped['dist_events'] = grouped['dist_events'].clip(lower=1e-10)
    grouped['dist_non_events'] = grouped['dist_non_events'].clip(lower=1e-10)
    
    grouped['WOE'] = np.log(grouped['dist_events'] / grouped['dist_non_events'])
    grouped['IV'] = (grouped['dist_events'] - grouped['dist_non_events']) * grouped['WOE']
    
    iv = grouped['IV'].sum()
    return grouped['WOE'].to_dict(), iv

# Calculate WOE & IV and apply encoding
iv_summary = {}

for col in woe_cols:
    if col in train.columns:
        woe_map, iv = calculate_woe_iv(train, col, 'isFraud')
        iv_summary[col] = iv
        
        # Apply WOE encoding - train
        train[col + '_WOE'] = train[col].astype(str).fillna('Missing').map(woe_map).fillna(0)
        
        # Apply WOE encoding - test (unseen categories get 0)
        test[col + '_WOE'] = test[col].astype(str).fillna('Missing').map(woe_map).fillna(0)
        
        # Drop original column - replaced by WOE encoded version
        train.drop(columns=[col], inplace=True)
        test.drop(columns=[col], inplace=True)
        
        print(f"✅ {col} → IV: {iv:.4f} → WOE encoded")

print("\n📊 IV Summary (Feature Importance):")
print("-" * 35)
iv_df = pd.DataFrame.from_dict(iv_summary, orient='index', columns=['IV'])
iv_df = iv_df.sort_values('IV', ascending=False)
for col, row in iv_df.iterrows():
    iv = row['IV']
    if iv < 0.02:
        strength = "🔴 Useless"
    elif iv < 0.1:
        strength = "🟡 Weak"
    elif iv < 0.3:
        strength = "🟠 Medium"
    elif iv < 0.5:
        strength = "🟢 Strong"
    else:
        strength = "⚠️ Suspicious"
    print(f"{col}: {iv:.4f} → {strength}")

train = train.copy()
test = test.copy()

In [ ]:
train.head()

In [ ]:
object_cols = train.select_dtypes(include='object').columns
for col in object_cols:
    print(f"{col}: {train[col].nunique()} unique values")

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

object_cols = train.select_dtypes(include='object').columns

ohe_cols = [col for col in object_cols if train[col].nunique() <= 5]
le_cols = [col for col in object_cols if train[col].nunique() > 5]

print(f"OHE columns ({len(ohe_cols)}): {ohe_cols}")
print(" ")
print(f"LabelEncoder columns ({len(le_cols)}): {le_cols}")

# Apply OHE
print(f"\nBefore OHE - Train shape: {train.shape}")
train = pd.get_dummies(train, columns=ohe_cols)
test = test.reindex(columns=train.columns, fill_value=0)
print(f"After OHE - Train shape: {train.shape}")
print(" ")

# Apply Label Encoding
for col in le_cols:
    if col in train.columns:
        le = LabelEncoder()
        le.fit(pd.concat([train[col], test[col]]).astype(str))
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
        print(f"Label encoded: {col}")

print("✅ Encoding complete!")

In [ ]:
y = train['isFraud']
X = train.drop(columns=['isFraud', 'TransactionID'])

results_df = pd.DataFrame(columns=['Run Name', 'Estimators', 'Max Depth', 'Learning Rate', 'AUC Score', 'Notes'])

In [ ]:
# Fill NaN

# Split
train_idx = int(len(X) * 0.70)
val_idx = int(len(X) * 0.85)

X_train, y_train = X.iloc[:train_idx].copy(), y.iloc[:train_idx].copy()
X_val, y_val = X.iloc[train_idx:val_idx].copy(), y.iloc[train_idx:val_idx].copy()
X_local_test, y_local_test = X.iloc[val_idx:].copy(), y.iloc[val_idx:].copy()

for col in X.columns:
    if X_train[col].dtype in ['int32', 'float32', 'float64']:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col] = X_val[col].fillna(median_val)
        X_local_test[col] = X_local_test[col].fillna(median_val)

print("✅ Split and NaN filling complete!")

# Models

In [ ]:
!pip install imbalanced-learn -q
!pip install shap -q

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import mlflow
import mlflow.sklearn
import gc

RUN_RF = True  

if RUN_RF:
    
    rf_params = {
        'n_estimators': 500,
        'max_depth': 12,          
        'min_samples_split': 50,   
        'min_samples_leaf': 50,    
        'max_features': 'sqrt',
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }

    with mlflow.start_run(run_name="RandomForest_BestModel"):
        mlflow.log_params(rf_params)

        model_rf = RandomForestClassifier(**rf_params)
        model_rf.fit(X_train, y_train)

        train_probs = model_rf.predict_proba(X_train)[:, 1]
        val_probs = model_rf.predict_proba(X_val)[:, 1]
        test_probs = model_rf.predict_proba(X_local_test)[:, 1]

        test_preds = model_rf.predict(X_local_test)

        train_auc = roc_auc_score(y_train, train_probs)
        val_auc = roc_auc_score(y_val, val_probs)
        test_auc = roc_auc_score(y_local_test, test_probs)

        test_precision = precision_score(y_local_test, test_preds)
        test_recall = recall_score(y_local_test, test_preds)
        test_f1 = f1_score(y_local_test, test_preds)

        mlflow.log_metric("Train_AUC", train_auc)
        mlflow.log_metric("Validation_AUC", val_auc)
        mlflow.log_metric("Test_AUC", test_auc)
        mlflow.log_metric("Precision", test_precision)
        mlflow.log_metric("Recall", test_recall)
        mlflow.log_metric("F1", test_f1)

        print("\n" + "="*40)
        print("🎯 Final Results !!!")
        print("="*40)
        print(f"📊 Train AUC: {train_auc:.5f}")
        print(f"📊 Validation AUC: {val_auc:.5f}")
        print(f"📊 Local Test AUC: {test_auc:.5f}")
        print("-" * 40)
        print(f"🔍 Test Precision: {test_precision:.5f}")
        print(f"🎣 Test Recall: {test_recall:.5f}")
        print(f"⚖️ Test F1-Score: {test_f1:.5f}")
        print("="*40)

        mlflow.sklearn.log_model(model_rf, "rf_baseline_model")

    gc.collect()
else:
    print("⏭️ Random Forest Skipped !")

In [ ]:
import shap
import matplotlib.pyplot as plt

X_shap_sample = X_train.sample(500, random_state=42)

# Calculate SHAP values
explainer = shap.TreeExplainer(model_rf)
shap_values = explainer.shap_values(X_shap_sample)

# For RandomForest binary classification, shap_values is a list of 2 arrays
# Index [1] = fraud class (what we care about)

if isinstance(shap_values, list):
    shap_values_fraud = shap_values[1]  # old SHAP format
else:
    shap_values_fraud = shap_values[:, :, 1]  # new SHAP format

print(f"Shape after fix: {shap_values_fraud.shape}")

print("✅ SHAP values calculated!")
print(f"Shape: {shap_values_fraud.shape}")

# Plot 1 - Summary Plot (shows direction and magnitude)
plt.figure()
shap.summary_plot(shap_values_fraud, X_shap_sample, max_display=20, show=False)
plt.title("SHAP Summary Plot - Top 20 Features", fontsize=14)
plt.tight_layout()
plt.savefig("shap_summary_plot.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 2 - Bar Plot (simple importance ranking)
plt.figure()
shap.summary_plot(shap_values_fraud, X_shap_sample, plot_type="bar", max_display=20, show=False)
plt.title("SHAP Feature Importance - Top 20 Features", fontsize=14)
plt.tight_layout()
plt.savefig("shap_bar_plot.png", dpi=150, bbox_inches='tight')
plt.show()

with mlflow.start_run(run_name="RandomForest_SHAP_Analysis"):
    mlflow.log_artifact("shap_summary_plot.png")
    mlflow.log_artifact("shap_bar_plot.png")
    print("✅ SHAP plots logged to MLflow!")

In [ ]:
run_balanced = False

if run_balanced:
    from imblearn.ensemble import BalancedRandomForestClassifier
    from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
    import mlflow
    import mlflow.sklearn
    import gc


    brf_params = {
        'n_estimators': 500,
        'max_depth': 15,
        'min_samples_split': 20,
        'min_samples_leaf': 20,
        'max_features': 'sqrt',
        'bootstrap': False,
        'random_state': 42,
        'n_jobs': -1
    }

    with mlflow.start_run(run_name="BalancedRandomForest_Tuned"):
        mlflow.log_params(brf_params)

        model_brf = BalancedRandomForestClassifier(**brf_params)
        model_brf.fit(X_train, y_train)

        train_probs = model_brf.predict_proba(X_train)[:, 1]
        val_probs = model_brf.predict_proba(X_val)[:, 1]
        test_probs = model_brf.predict_proba(X_local_test)[:, 1]

        test_preds = model_brf.predict(X_local_test)

        train_auc = roc_auc_score(y_train, train_probs)
        val_auc = roc_auc_score(y_val, val_probs)
        test_auc = roc_auc_score(y_local_test, test_probs)

        test_precision = precision_score(y_local_test, test_preds)
        test_recall = recall_score(y_local_test, test_preds)
        test_f1 = f1_score(y_local_test, test_preds)

        mlflow.log_metric("Train_AUC", train_auc)
        mlflow.log_metric("Validation_AUC", val_auc)
        mlflow.log_metric("Test_AUC", test_auc)
        mlflow.log_metric("Precision", test_precision)
        mlflow.log_metric("Recall", test_recall)
        mlflow.log_metric("F1", test_f1)

        print("\n" + "="*40)
        print("🎯 Final Results !!!")
        print("="*40)
        print(f"📊 Train AUC: {train_auc:.5f}")
        print(f"📊 Validation AUC: {val_auc:.5f}")
        print(f"📊 Local Test AUC: {test_auc:.5f}")
        print("-" * 40)
        print(f"🔍 Test Precision: {test_precision:.5f}")
        print(f"🎣 Test Recall: {test_recall:.5f}")
        print(f"⚖️ Test F1-Score: {test_f1:.5f}")
        print("="*40)

        mlflow.sklearn.log_model(model_brf, "brf_baseline_model")

    del model_brf
    gc.collect()